Phase 0

Phase 1 : Séparer les données proprement, train / validation / test

In [62]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np


def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    if val_size == 0:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        X_val = X_test[:0]
        y_val = y_test[:0]
        print(f"Train : {len(X_train)} | Validation : {len(X_val)} | Test : {len(X_test)}")
        return X_train, X_val, X_test, y_train, y_val, y_test

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    val_size_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size_adjusted, random_state=random_state, stratify=y_temp
    )

    print(f"Train : {len(X_train)} | Validation : {len(X_val)} | Test : {len(X_test)}")
    t_train, t_val, t_test = np.mean(y_train), np.mean(y_val), np.mean(y_test)
    stratif_ok = max(t_train, t_val, t_test) - min(t_train, t_val, t_test) < 0.02
    print(f"Taux classe positive — train: {t_train:.3f} | val: {t_val:.3f} | test: {t_test:.3f}")
    print(f"Répartition des classes conservée dans chaque jeu : {'oui' if stratif_ok else 'non'}")
    return X_train, X_val, X_test, y_train, y_val, y_test

In [63]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("CAS NORMAL")
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X, y)
assert len(X_train) + len(X_val) + len(X_test) == len(X)
print("Somme des tailles = total : OK")

print("\nCAS LIMITE — val_size=0")
split_train_val_test(X, y, val_size=0)

print("\nCAS ADVERSARIAL — 95/5")
y_desq = pd.Series([0]*950 + [1]*50)
X_desq = pd.DataFrame(np.random.randn(1000, 5))
split_train_val_test(X_desq, y_desq)

CAS NORMAL
Train : 341 | Validation : 114 | Test : 114
Taux classe positive — train: 0.628 | val: 0.623 | test: 0.632
Répartition des classes conservée dans chaque jeu : oui
Somme des tailles = total : OK

CAS LIMITE — val_size=0
Train : 455 | Validation : 0 | Test : 114

CAS ADVERSARIAL — 95/5
Train : 600 | Validation : 200 | Test : 200
Taux classe positive — train: 0.050 | val: 0.050 | test: 0.050
Répartition des classes conservée dans chaque jeu : oui


(            0         1         2         3         4
 538  1.230858 -0.434730  0.963528  0.376758 -2.765909
 872  0.241213 -0.800220 -2.004832 -0.114387  1.055499
 51  -1.089098 -1.444081 -0.788937  0.573565 -0.877036
 69  -0.405096 -0.904081  1.876110 -0.126996  0.552715
 939 -2.167259 -0.267291  0.524818  2.177983  1.293380
 ..        ...       ...       ...       ...       ...
 932 -1.846762  1.315590 -0.438098  0.351859 -0.502678
 761  1.862562  1.226015  0.294079  0.636256  1.718779
 829  1.059247  0.573752  1.065027 -1.063231  1.548321
 857 -1.328096  0.844990  0.560873 -0.435259 -1.258603
 175 -0.171273 -0.332089 -1.269574  0.239884  0.675030
 
 [600 rows x 5 columns],
             0         1         2         3         4
 36  -1.232877 -0.188994  0.901850 -1.002431  0.176686
 601 -0.776281 -1.322783 -0.747981  0.086611 -0.919201
 694 -1.398332 -0.279445  0.095581 -0.999703 -0.095723
 786 -0.637463 -0.664098  0.189726 -0.432983  0.022358
 465  0.834240 -1.665919  1.686714  1.

Les résultat semble correct : 
569 en cas  normal 
cas limite  validation = 0 sans planter 
et 5% de class positive en cas adversariale



Phase 2 : Bootstrap et bagging, comprendre le rééchantillonnage

In [64]:
def bootstrap_scores(modele, X, y, n_iterations=30, random_state=42):
    """Évalue la stabilité d'un modèle par bootstrap.
    Pour chaque itération : tirer un échantillon AVEC REMISE de même taille
    que X, entraîner, évaluer sur les points NON tirés (out-of-bag).
    Doit renvoyer la liste des scores et afficher moyenne et écart-type.
    """
    rng = np.random.default_rng(random_state)
    scores = []
    n = len(X)
 
    for i in range(n_iterations):
        # tirer des indices avec remise
        indices_bootstrap = rng.choice(n, size=n, replace=True)
 
        # indices out-of-bag : points jamais tirés dans cet échantillon
        # ils n'ont pas servi à l'entraînement : jeu de test gratuit
        indices_oob = np.array(list(set(range(n)) - set(indices_bootstrap)))
 
        # cas rare : OOB vide → on skip cette itération
        if len(indices_oob) == 0:
            print(f"itération {i} : OOB vide, skippée")
            continue
 
        X_boot, y_boot = X[indices_bootstrap], y[indices_bootstrap]
        X_oob,  y_oob  = X[indices_oob],  y[indices_oob]
 
        # entraîner sur l'échantillon, scorer sur l'out-of-bag
        modele.fit(X_boot, y_boot)
        score = modele.score(X_oob, y_oob)
        scores.append(score)
 
    scores = np.array(scores)
    print(f"Score moyen sur {len(scores)} bootstraps : {scores.mean():.3f} (± {scores.std():.3f})")
    return scores

In [65]:
from sklearn.preprocessing import StandardScaler
 
# scaling d'abord (bonne pratique)
scaler_boot = StandardScaler()
X_scaled = scaler_boot.fit_transform(X)
 
modele_boot = RandomForestClassifier(n_estimators=50, random_state=42)
 
# === CAS NORMAL ===
print("=== CAS NORMAL (30 bootstraps) ===\n")
scores_boot = bootstrap_scores(modele_boot, X_scaled, y, n_iterations=30)
 
# === CAS LIMITE : replace=False (sans remise) ===
print("\n=== CAS LIMITE (sans remise replace=False) ===\n")
def bootstrap_scores_sans_remise(modele, X, y, n_iterations=30, random_state=42):
    rng = np.random.default_rng(random_state)
    scores = []
    n = len(X)
    for i in range(n_iterations):
        # sans remise = simple mélange, pas un vrai bootstrap
        indices = rng.choice(n, size=n, replace=False)
        milieu  = n // 2
        X_boot, y_boot = X[indices[:milieu]], y[indices[:milieu]]
        X_oob,  y_oob  = X[indices[milieu:]], y[indices[milieu:]]
        modele.fit(X_boot, y_boot)
        scores.append(modele.score(X_oob, y_oob))
    scores = np.array(scores)
    print(f"Score moyen SANS remise : {scores.mean():.3f} (± {scores.std():.3f})")
    return scores
 
scores_sans = bootstrap_scores_sans_remise(modele_boot, X_scaled, y)
print(f"\n→ Écart-type AVEC remise : {scores_boot.std():.3f}")
print(f"  Écart-type SANS remise : {scores_sans.std():.3f}")
print("  Sans remise = simple mélange, pas de vraie variété bootstrap")
 
# === CAS LIMITE : n_iterations=1 ===
print("\n=== CAS LIMITE (n_iterations=1) ===\n")
scores_1 = bootstrap_scores(modele_boot, X_scaled, y, n_iterations=1)
print("→ Un seul score, écart-type=0.000 : aucune stabilité mesurable")

=== CAS NORMAL (30 bootstraps) ===

Score moyen sur 30 bootstraps : 0.960 (± 0.015)

=== CAS LIMITE (sans remise replace=False) ===

Score moyen SANS remise : 0.955 (± 0.013)

→ Écart-type AVEC remise : 0.015
  Écart-type SANS remise : 0.013
  Sans remise = simple mélange, pas de vraie variété bootstrap

=== CAS LIMITE (n_iterations=1) ===

Score moyen sur 1 bootstraps : 0.910 (± 0.000)
→ Un seul score, écart-type=0.000 : aucune stabilité mesurable


Phase : 3 La validation croisée k-fold

Cas normal entre 0.96 et 0.015 indique de bonnes performances
cas limite val_size = 0 est géré correctement sans provoquer d'erreur
Cas déséquilibré 95/5 garenti une evolution fiable meme avec donnée déséquilibré
Bootstrap avec et sans remise : les deux approches donnent des performances proches, mais le bootstrap avec remise apporte une plus grande diversité d'échantillons. Sans remise, on obtient essentiellement un simple réarrangement des données et non un véritable bootstrap
Cas limlite : Il n'est pas possible d'évaluer la stabilité du modèle puis que l'écart type est nul 

résumé : mes fonction sont robuste et permettent d'obtenir des estimation fiable 

In [66]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

def evaluer_en_cross_val(modele, X, y, k=5):

    cv = StratifiedKFold(
        n_splits=k,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        modele,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    print("Scores par fold :", scores)

    moyenne = np.mean(scores)
    ecart_type = np.std(scores)

    print(f"Moyenne : {moyenne:.3f} | Écart-type : {ecart_type:.3f}")

    if ecart_type < 0.02:
        print("→ modèle stable")
    else:
        print("→ modèle instable")

    return scores

In [67]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier

# Chargement des données
data = load_breast_cancer()
X = data.data
y = data.target

# Création du modèle
modele = RandomForestClassifier(random_state=42)

# Validation croisée
evaluer_en_cross_val(modele, X, y, k=5)

Scores par fold : [0.96491228 0.93859649 0.95614035 0.94736842 0.97345133]
Moyenne : 0.956 | Écart-type : 0.012
→ modèle stable


array([0.96491228, 0.93859649, 0.95614035, 0.94736842, 0.97345133])

In [68]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

def evaluer_en_cross_val(modele, X, y, k=5):

    cv = StratifiedKFold(
        n_splits=k,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        modele,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    print("Scores par fold :", scores)

    moyenne = np.mean(scores)
    ecart_type = np.std(scores)

    print(f"Moyenne : {moyenne:.3f} | Écart-type : {ecart_type:.3f}")

    if ecart_type < 0.02:
        print("→ modèle stable")
    else:
        print("→ modèle instable")

    return scores

les scores fold sont proche entre 0.93 et 0.97, la moyenne à un très bon score 0.956
l'écart-type démontre une faible dispersion ( ecart moyen autour de la moyenne)

en clair, le faible écart type indique que le modèle est robuste au variation des donnée et dépend nuellement de la composition des folds 
